# 9.10 Bifurcated + sparse simulator — holes on vs off

Runs **`model5` against the Base Decipher control** — a two-arm head-to-head — on
`simulated_data_pipeline_924.py`, which is `simulated_data_pipeline_bifurcation_sep18.py`
plus optional chunk/hole sampling of `latent_t`.

The sweep varies **one new axis**: `n_holes in [0, 3]`. `n_holes=0` produces data that is
bit-identical to sep18's, so it is the matched baseline for the sparse arm — not a separate
experiment.

## What "holes" are, and what they are not

The hole machinery is carried over character-for-character from
`Data/Simulated Data/Simulated Data Generation/simulations_original.py`, which is an
**md5-identical** copy of the Decipher author's `decipher_reproducibility/simulations.py`.

**Holes are gaps in pseudotime.** The trajectory is cut into alternating dense and sparse
chunks, and `latent_t` is sampled so the sparse chunks get `hole_density` times as many cells
per unit length. They model uneven cell density along a trajectory.

**Holes are NOT sparsity in the count matrix.** Nothing adds dropout, zero-inflation or masking
to `X`. `adata.X` stays a dense integer count matrix either way. If you are looking for
technical dropout, this is not it.

Two settings matter:

- `hole_size = 1` — **keep this**. It is the only value the author ever used, and the only
  value at which upstream's `chunk_offset = np.cumsum(chunk_size) - chunk_size[0]` is a correct
  exclusive prefix sum. At `hole_size=2` the chunks overlap and leave uncovered gaps, producing
  genuinely *empty* pseudotime regions rather than thin ones. `_validate_holes` warns if you
  change it. The upstream bug is preserved rather than repaired, so this stays faithful.
- `hole_density = 0.05` — the author's own setting in `simulations.py`.

## Why gene-mode, and why only the Set 3 arms

`batch_mode="genes"` puts the batch effect on `pre_x` **after** `@ W`, and adds nothing to
`z_mean` — so `latent_z` carries no batch information. That is the `p(x|z,b)` generative process
the Set 3 arms (`model4`/`model5`/`model6`, `concat -> x`) assume.

Set 1 / Set 2 arms assume the batch lives in `z` and should be run against `batch_mode="z"` on
their own data. Per `0918_sweep_3models_bifurcation.py`, arms trained on different `batch_mode`
data are scored on **different datasets** and their raw rho is not comparable. Compare each arm
to the `Base Decipher` arm sharing its regime, then compare those deltas.

**Scope: two arms.** `model5` only, against `Base Decipher`. `model5` is the Set 3 arm with
`batch_conditioning="decoder_encoder"` — batch enters both the encoder and the decoder — which
makes it the direct `concat -> x` counterpart to `model2` in Set 1. `model4` (`decoder_only`)
and `model6` (`mean_field_v=True`) are left out to keep this first sparse run small; add them
back to `models` and `colors` below when the two-arm result is understood.

## Setup

In [ ]:
import sys, os, importlib, inspect
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

# CHANGED 2026-09-24 -- was a hardcoded absolute path, which only existed on Alan's laptop:
#     REPO = "/Users/alanxie/CodingProjects/Columbia/Azizi Lab/Spatial Nut Carcinoma/decipher-bc-methods"
# Replaced so the same notebook runs unmodified on the VM.

_MARKERS = ("decipher_models", "Data/Simulated Data/Simulated Data Generation")

def find_repo_root(start=None, markers=_MARKERS):
    """Locate the decipher-bc-methods root without hardcoding a machine-specific path.

    $DECIPHER_BC_REPO wins if set; otherwise walk up from cwd until a directory contains every
    marker. Both markers are tracked in git, so a fresh clone has them. Raises rather than
    guessing -- a wrong root would not fail here, it would import a DIFFERENT pipeline file or
    blow up much later with an unrelated message.
    """
    env = os.environ.get("DECIPHER_BC_REPO")
    if env:
        root = Path(env).expanduser().resolve()
        missing = [m for m in markers if not (root / m).exists()]
        if missing:
            raise RuntimeError(
                f"DECIPHER_BC_REPO={env!r} is set but is missing: {missing}. "
                f"Point it at the decipher-bc-methods root, or unset it to auto-detect."
            )
        return str(root)

    here = Path(start or os.getcwd()).resolve()
    for cand in (here, *here.parents):
        if all((cand / m).exists() for m in markers):
            return str(cand)
    raise RuntimeError(
        f"could not find the decipher-bc-methods root above {here}. "
        f"Looked for all of {markers}. Set DECIPHER_BC_REPO to the repo root, e.g.\n"
        f"    os.environ['DECIPHER_BC_REPO'] = '/home/ubuntu/decipher-bc-methods'"
    )

REPO = find_repo_root()
PIPELINE_DIR = os.path.join(REPO, "Data/Simulated Data/Simulated Data Generation")
if PIPELINE_DIR not in sys.path:
    sys.path.insert(0, PIPELINE_DIR)
print("REPO        :", REPO)
print("PIPELINE_DIR:", PIPELINE_DIR)

# NEW in 9.10: the 924 pipeline (sep18 + holes), not aug1 and not sep18.
from simulated_data_pipeline_924 import (
    train_and_compute_rho_r2_bifurcation,
    shift_magnitudes_multivariate_bifurcation,
    _data_tag,
)

# Tags every output this notebook writes, INCLUDING the trained h5ad filenames, so a second
# sweep on the same calendar day cannot silently overwrite this one's files.
NOTEBOOK_TAG = "9.10"
BATCH_MODE = "genes"      # matches the Set 3 (concat -> x) generative assumption

## Sweep config

In [ ]:
# ---- sweep config ----
shift_sigmas = [0.1, 0.5, 1.0, 2.0]
seeds = [3, 4]            # simulation seed: which DATASET
decipher_seeds = [1]      # training seed: SVI init only
beta = 0.1
n_z_dims = 3              # NOT the DecipherConfig default of 10; the pipeline overrides it
n_samples = 500
n_genes = 200
biological_sigma = 0.1

# bifurcation, inherited from sep18
branching_t = 0.3
branch_prob = 0.5

# ---- THE NEW AXIS ----
# (label, hole_size, n_holes, hole_density)
#   "nohole" reproduces sep18 bit-for-bit and is the matched baseline.
#   "holes3" is the Decipher author's own setting from simulations.py.
HOLE_CONFIGS = [
    ("nohole", 1, 0, 0.0),
    ("holes3", 1, 3, 0.05),
]

# NARROWED to a two-arm head-to-head. model4 / model6 commented out rather than deleted so
# the diff shows what was dropped; uncomment both lines to widen back to the full Set 3.
models = {
    "Base Decipher":          "native",
    # "Set3 concat->x  model4": "model4",
    "Set3 concat->x  model5": "model5",
    # "Set3 concat->x  model6": "model6",
}
colors = {
    "Base Decipher":          "#545151",
    "Set3 concat->x  model4": "#1565C0",
    "Set3 concat->x  model5": "#EF6C00",
    "Set3 concat->x  model6": "#2E7D32",
}

today = datetime.now().strftime("%m%d")
log_dir = os.path.join(PIPELINE_DIR, "..", "Simulated Adata", "shift_sigma_sweep_bifurcation_sparse", today)
os.makedirs(log_dir, exist_ok=True)
csv_path = os.path.join(log_dir, f"sweep_log_{NOTEBOOK_TAG}_{shift_sigmas}_{seeds}.csv")

n_runs = len(models) * len(shift_sigmas) * len(seeds) * len(decipher_seeds) * len(HOLE_CONFIGS)
print(f"{len(models)} arms x {len(shift_sigmas)} sigmas x {len(seeds)} seeds "
      f"x {len(HOLE_CONFIGS)} hole configs = {n_runs} runs")
print("writing to:", csv_path)

## Pre-flight check — run this before the sweep

Two things worth confirming before spending the training time, both cheap because they only
touch the simulator:

1. **`n_holes=0` is bit-identical to sep18.** If this fails, the guard in
   `simulate_multivariate_bifurcation` is not doing its job and the baseline arm is not
   comparable to anything you ran before.
2. **`n_holes=3` actually thins pseudotime**, and thins it rather than emptying it.

In [ ]:
import warnings
import simulated_data_pipeline_bifurcation_sep18 as sep18

sim_kw = dict(n_batches=5, shift_sigma=1.0, n_samples=n_samples, n_genes=n_genes,
              n_z_dims=n_z_dims, biological_sigma=biological_sigma, seed=3,
              batch_mode=BATCH_MODE, branching_t=branching_t, branch_prob=branch_prob)

a_old = sep18.shift_magnitudes_multivariate_bifurcation(**sim_kw)
a_off = shift_magnitudes_multivariate_bifurcation(**sim_kw, hole_size=1, n_holes=0, hole_density=0.0)
a_on  = shift_magnitudes_multivariate_bifurcation(**sim_kw, hole_size=1, n_holes=3, hole_density=0.05)

same_X = np.array_equal(np.asarray(a_old.X), np.asarray(a_off.X))
same_t = np.array_equal(a_old.obs["latent_t"].values, a_off.obs["latent_t"].values)
print(f"CHECK 1  n_holes=0 vs sep18 -- X identical: {same_X}, latent_t identical: {same_t}")
assert same_X and same_t, "holes-off is NOT bit-identical to sep18; the n_holes guard is broken"

h_on,  _ = np.histogram(a_on.obs["latent_t"],  bins=14, range=(0, 1))
h_off, _ = np.histogram(a_off.obs["latent_t"], bins=14, range=(0, 1))
print(f"CHECK 2  latent_t, 14 bins over [0,1], {a_on.n_obs} cells")
print(f"  holes OFF: {list(h_off)}")
print(f"  holes ON : {list(h_on)}")
print(f"  thinnest bin  off={h_off.min()}  on={h_on.min()}   empty bins on={int((h_on == 0).sum())}")
assert h_on.min() > 0, "a bin is EMPTY -- check hole_size is 1, not 2"

fig, ax = plt.subplots(figsize=(7.5, 3.6))
edges = np.linspace(0, 1, 15)
ax.stairs(h_off, edges, label="n_holes=0 (= sep18)", linewidth=2)
ax.stairs(h_on,  edges, label="n_holes=3, hole_density=0.05", linewidth=2)
ax.axvline(branching_t, color="k", linestyle=":", linewidth=1.2, label=f"branching_t={branching_t}")
ax.set_xlabel("latent_t"); ax.set_ylabel("cells"); ax.set_title("Pseudotime density, holes off vs on")
ax.spines[["top", "right"]].set_visible(False); ax.legend(frameon=False, fontsize=8)
fig.tight_layout(); fig.savefig(os.path.join(log_dir, f"latent_t_density_{NOTEBOOK_TAG}.png"), dpi=200)
plt.show()

# Does the fork still have cells around it? A hole straddling branching_t would leave nothing to
# learn the bifurcation from.
for lbl, ad_ in [("off", a_off), ("on", a_on)]:
    t = ad_.obs["latent_t"].values
    near = int(((t > branching_t - 0.05) & (t < branching_t + 0.05)).sum())
    print(f"  holes {lbl:3s}: {near} cells within +/-0.05 of branching_t, "
          f"{int((t > branching_t).sum())} post-fork")

## Run sweep

In [ ]:
run_log = []
for hole_label, hole_size, n_holes, hole_density in HOLE_CONFIGS:
    for name, model in models.items():
        for shift_sigma in shift_sigmas:
            for sd in seeds:
                for decipher_seed in decipher_seeds:
                    record = {
                        "model": name, "n_z_dims": n_z_dims,
                        "shift_sigma": shift_sigma, "seed": sd,
                        "decipher_seed": decipher_seed, "beta": beta,
                        "batch_mode": BATCH_MODE,
                        "branching_t": branching_t, "branch_prob": branch_prob,
                        "hole_label": hole_label, "hole_size": hole_size,
                        "n_holes": n_holes, "hole_density": hole_density,
                    }
                    try:
                        (rho, rho_plus, rho_minus, branch_asw, trained_path,
                         r2_overall, r2_per_gene_median, _) = train_and_compute_rho_r2_bifurcation(
                            model, decipher_seed,
                            shift_sigma=shift_sigma,
                            n_samples=n_samples,
                            n_genes=n_genes,
                            biological_sigma=biological_sigma,
                            seed=sd,
                            n_z_dims=n_z_dims,
                            beta=beta,
                            notebook_tag=NOTEBOOK_TAG,   # keeps these h5ads out of sep18's names
                            batch_mode=BATCH_MODE,
                            branching_t=branching_t,
                            branch_prob=branch_prob,
                            hole_size=hole_size,         # NEW in 9.10
                            n_holes=n_holes,
                            hole_density=hole_density,
                        )
                        record.update({
                            "rho": rho, "rho_plus": rho_plus, "rho_minus": rho_minus,
                            "branch_asw": branch_asw, "trained_h5ad": trained_path,
                            "r2_overall": r2_overall,
                            "r2_per_gene_median": r2_per_gene_median,
                            "error": None,
                        })
                    except Exception as e:
                        print(f"[{name}] {hole_label} shift_sigma={shift_sigma} seed={sd} "
                              f"failed: {type(e).__name__}: {e}")
                        record.update({
                            "rho": np.nan, "rho_plus": np.nan, "rho_minus": np.nan,
                            "branch_asw": np.nan, "trained_h5ad": None,
                            "r2_overall": np.nan, "r2_per_gene_median": np.nan,
                            "error": f"{type(e).__name__}: {e}",
                        })
                    run_log.append(record)
                    pd.DataFrame(run_log).to_csv(csv_path, index=False)

run_log_df = pd.DataFrame(run_log)
print(f"done. {run_log_df['error'].isna().sum()} ok, {run_log_df['error'].notna().sum()} failed")
run_log_df

## Filename check

Holes are a new sweep axis, and this pipeline follows sep18's stated RULE: *every axis the sweep
varies must appear in the output filename.* If two hole configs collided on one path, the loser's
results would vanish with no error — which has already happened twice in this codebase.

In [ ]:
ok_df = run_log_df[run_log_df["error"].isna()]
paths = ok_df["trained_h5ad"]
print(f"{len(paths)} runs, {paths.nunique()} distinct h5ad paths")
assert len(paths) == paths.nunique(), "COLLISION: two runs wrote the same h5ad path"

print("\ntag fragments actually produced:")
for lbl, hs, nh, hd in HOLE_CONFIGS:
    print(f"  {lbl:7s} -> {_data_tag(branching_t, BATCH_MODE, nh, hs, hd)}")
print("\nexample filename:")
print(" ", os.path.basename(paths.iloc[0]))

## Pseudotime recovery — the headline comparison

Solid = holes on, dashed = holes off. The question is not which arm is highest; it is whether the
gap between an arm and `Base Decipher` **changes** when the trajectory becomes sparse.

In [ ]:
run_log_df = run_log_df[run_log_df["error"].isna()].copy()
sigmas_sorted = sorted(run_log_df["shift_sigma"].unique())
x_pos = np.arange(len(sigmas_sorted))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, (hole_label, *_rest) in zip(axes, HOLE_CONFIGS):
    sub_h = run_log_df[run_log_df["hole_label"] == hole_label]
    for name in models:
        sub = sub_h[sub_h["model"] == name]
        if sub.empty:
            continue
        stats = sub.groupby("shift_sigma")["rho"].agg(["mean", "std"]).reindex(sigmas_sorted)
        mean, sd_ = stats["mean"].to_numpy(), stats["std"].to_numpy()
        ax.plot(x_pos, mean, marker="o", color=colors[name], label=name, linewidth=2)
        ax.fill_between(x_pos, mean - sd_, mean + sd_, color=colors[name], alpha=0.12)
    ax.set_xticks(x_pos); ax.set_xticklabels([str(s) for s in sigmas_sorted])
    ax.set_xlabel(r"batch-shift noise $\sigma$")
    ax.set_title(f"{hole_label}")
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel(r"Spearman $|\rho|$ vs latent_t")
axes[0].legend(frameon=False, fontsize=8)
fig.suptitle(f"Pseudotime recovery, gene-mode + bifurcation (seeds={seeds}, beta={beta})", y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(log_dir, f"rho_{NOTEBOOK_TAG}.png"), dpi=200, bbox_inches="tight")
plt.show()

run_log_df.groupby(["hole_label", "model", "shift_sigma"])["rho"].agg(["mean", "std"]).round(3)

### Delta against the matched control

Each arm minus `Base Decipher` at the same (hole config, sigma, seed). This is the comparison the
sweep driver says is valid — the raw rho columns above are on different datasets between the two
panels, so their absolute heights are not comparable.

In [ ]:
base = (run_log_df[run_log_df["model"] == "Base Decipher"]
        .set_index(["hole_label", "shift_sigma", "seed", "decipher_seed"])["rho"])
d = run_log_df.join(
    base.rename("rho_base"), on=["hole_label", "shift_sigma", "seed", "decipher_seed"])
d["rho_delta"] = d["rho"] - d["rho_base"]
d = d[d["model"] != "Base Decipher"]

fig, ax = plt.subplots(figsize=(8, 4.5))
for name in [m for m in models if m != "Base Decipher"]:
    for hole_label, ls in zip([h[0] for h in HOLE_CONFIGS], ["--", "-"]):
        sub = d[(d["model"] == name) & (d["hole_label"] == hole_label)]
        if sub.empty:
            continue
        stats = sub.groupby("shift_sigma")["rho_delta"].mean().reindex(sigmas_sorted)
        ax.plot(x_pos, stats.to_numpy(), marker="o", linestyle=ls,
                color=colors[name], linewidth=2, label=f"{name} ({hole_label})")
ax.axhline(0, color="k", linewidth=0.8)
ax.set_xticks(x_pos); ax.set_xticklabels([str(s) for s in sigmas_sorted])
ax.set_xlabel(r"batch-shift noise $\sigma$")
ax.set_ylabel(r"$\rho$ - $\rho_{\rm Base\ Decipher}$")
ax.set_title("Improvement over the no-correction control (dashed = holes off, solid = on)")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, fontsize=7)
fig.tight_layout()
fig.savefig(os.path.join(log_dir, f"rho_delta_{NOTEBOOK_TAG}.png"), dpi=200)
plt.show()

d.groupby(["hole_label", "model"])["rho_delta"].agg(["mean", "std"]).round(3)

## Branch metrics

Inherited from sep18 and specific to the bifurcation: `rho_plus` / `rho_minus` score pseudotime
**along one arm at a time**, and `branch_asw` is the batch silhouette on `decipher_v` restricted
to post-fork cells. Holes thin the arms as well as the trunk, so these are where a sparsity
effect should show up first.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))
for ax, metric, title in zip(
        axes, ["rho_plus", "rho_minus", "branch_asw"],
        [r"$\rho$ along arm +1", r"$\rho$ along arm -1", "branch ASW (post-fork)"]):
    for name in models:
        for hole_label, ls in zip([h[0] for h in HOLE_CONFIGS], ["--", "-"]):
            sub = run_log_df[(run_log_df["model"] == name) &
                             (run_log_df["hole_label"] == hole_label)]
            if sub.empty:
                continue
            stats = sub.groupby("shift_sigma")[metric].mean().reindex(sigmas_sorted)
            ax.plot(x_pos, stats.to_numpy(), marker="o", linestyle=ls,
                    color=colors[name], linewidth=1.8)
    ax.set_xticks(x_pos); ax.set_xticklabels([str(s) for s in sigmas_sorted])
    ax.set_xlabel(r"$\sigma$"); ax.set_title(title)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("value")
fig.suptitle("Branch metrics (dashed = holes off, solid = holes on)", y=1.03)
fig.tight_layout()
fig.savefig(os.path.join(log_dir, f"branch_metrics_{NOTEBOOK_TAG}.png"), dpi=200, bbox_inches="tight")
plt.show()

run_log_df.groupby(["hole_label", "model"])[["rho_plus", "rho_minus", "branch_asw"]].mean().round(3)

## The zero-batch ceiling

Gene-mode writes `layers["counts_nobatch"]`: the exact dataset this would have been with
`gene_shift = 0` and everything else held fixed. Its batch silhouette is the floor a perfect
correction could reach — not zero, because finite samples leave some separation.

In [ ]:
import scanpy as sc
from sklearn.metrics import silhouette_score

def _dense_X(adata):
    X = adata.X
    return np.asarray(X.todense()) if hasattr(X, "todense") else np.asarray(X)

ceiling_records = []
for _, row in run_log_df.iterrows():
    adata = sc.read_h5ad(row["trained_h5ad"])
    codes = adata.obs["batch"].astype("category").cat.codes.values
    rec = {k: row[k] for k in ["model", "shift_sigma", "seed", "hole_label"]}
    rec["sil_counts"] = silhouette_score(np.log1p(_dense_X(adata)), codes)
    if "counts_nobatch" in adata.layers:
        clean = np.asarray(adata.layers["counts_nobatch"], dtype=float)
        rec["sil_counts_nobatch"] = silhouette_score(np.log1p(clean), codes)
    else:
        rec["sil_counts_nobatch"] = np.nan
    rec["sil_decipher_z"] = silhouette_score(adata.obsm["decipher_z"], codes)
    rec["sil_decipher_v"] = silhouette_score(adata.obsm["decipher_v"], codes)
    ceiling_records.append(rec)

ceiling_df = pd.DataFrame(ceiling_records)
ceiling_df.to_csv(os.path.join(log_dir, f"ceiling_{NOTEBOOK_TAG}.csv"), index=False)

print("Batch silhouette. Lower = better mixed.")
print("  sil_counts         : the data as given to the model")
print("  sil_counts_nobatch : THE CEILING -- same data with gene_shift = 0")
ceiling_df.groupby(["hole_label", "model", "shift_sigma"])[
    ["sil_counts", "sil_counts_nobatch", "sil_decipher_z", "sil_decipher_v"]
].mean().round(4)

## Per-sigma, per-model detail

In [ ]:
model_names = list(models.keys())
seed_for_detail = seeds[0]

def load_run(df, shift_sigma, model_name, hole_label, seed=None):
    sub = df[(df["shift_sigma"] == shift_sigma)
             & (df["model"] == model_name)
             & (df["hole_label"] == hole_label)]
    if seed is not None:
        sub = sub[sub["seed"] == seed]
    row = sub.iloc[0]
    return row, sc.read_h5ad(row["trained_h5ad"])

### V-space: batch, true pseudotime, learned pseudotime

`decipher_v` is 2-D and `latent_v` is 2-D, so the ground-truth Y and the learned embedding live
in spaces of the same dimension. A working arm shows one Y with the batches overlaid, not five
separated copies.

In [ ]:
def plot_v_space(shift_sigma, hole_label, seed=seed_for_detail):
    for model_name in model_names:
        row, adata = load_run(run_log_df, shift_sigma, model_name, hole_label, seed=seed)
        sc.pl.embedding(adata, basis="decipher_v",
                        color=["batch", "latent_t", "branch_id", "decipher_time"])
        print(f"{model_name} | {hole_label} | sigma={shift_sigma} | seed={row['seed']} | "
              f"rho={row['rho']:.4f}  rho+={row['rho_plus']:.4f}  rho-={row['rho_minus']:.4f}")

#### Sigma = 0.1

In [ ]:
plot_v_space(0.1, "nohole")
plot_v_space(0.1, "holes3")

#### Sigma = 0.5

In [ ]:
plot_v_space(0.5, "nohole")
plot_v_space(0.5, "holes3")

#### Sigma = 1.0

In [ ]:
plot_v_space(1.0, "nohole")
plot_v_space(1.0, "holes3")

#### Sigma = 2.0

In [ ]:
plot_v_space(2.0, "nohole")
plot_v_space(2.0, "holes3")

### Z-space: learned `decipher_z` against the ceiling

In gene-mode the batch offset is added after `@ W`, so `latent_z` is batch-free by construction.
A batch silhouette near zero on `latent_z` is a sanity check that the simulator did what it
claims — not a result.

In [ ]:
def plot_z_space(shift_sigma, hole_label, seed=seed_for_detail):
    for model_name in model_names:
        row, adata = load_run(run_log_df, shift_sigma, model_name, hole_label, seed=seed)
        codes = adata.obs["batch"].astype("category").cat.codes.values

        sil_ceiling = (silhouette_score(
            np.log1p(np.asarray(adata.layers["counts_nobatch"], float)), codes)
            if "counts_nobatch" in adata.layers else float("nan"))
        sil_learned = silhouette_score(adata.obsm["decipher_z"], codes)

        print(f"{model_name} | {hole_label} | sigma={shift_sigma} | seed={seed}")
        print(f"    ceiling (counts_nobatch) batch silhouette = {sil_ceiling:.4f}")
        print(f"    learned decipher_z batch silhouette       = {sil_learned:.4f}")

        sc.pp.neighbors(adata, use_rep="decipher_z", key_added="dz_neighbors", random_state=42)
        sc.tl.umap(adata, neighbors_key="dz_neighbors", random_state=42)
        sc.pl.umap(adata, color=["batch", "latent_t", "branch_id"],
                   title=[f"{model_name} decipher_z, batch",
                          f"{model_name} decipher_z, latent_t",
                          f"{model_name} decipher_z, branch_id"])

#### Sigma = 0.1

In [ ]:
plot_z_space(0.1, "nohole")
plot_z_space(0.1, "holes3")

#### Sigma = 1.0

In [ ]:
plot_z_space(1.0, "nohole")
plot_z_space(1.0, "holes3")